# P2 D-256 StyleGAN2 — Colab Reproduction

FFHQ 256×256 from scratch, 14M images, bilinear→1024 wrapper for submission.

**전략:** 매 세션 GitHub fresh clone → Colab 로컬에서 학습 → ckpt는 Drive에 직접 저장 (Disconnect 시 자동 보존). 코드는 항상 git HEAD와 일치 → 재현성 보장.

**ETA:** A100 ~22h (14M images). 24h session 1회 disconnect 가능 → Resume 셀로 이어 학습.

## Drive 사전 준비

재현 시 다음 한 가지만 준비:
```
/MyDrive/osai/p2/train_50k_256.zip        ← 학습 데이터 (필수, 1.65GB)
/MyDrive/osai/p2/valid_10k_256.zip        ← FID 자기측정 (선택, 0.33GB)
```

그리고 Colab Secrets (왼쪽 사이드바 🔑)에 `WANDB_API_KEY` 등록 + Notebook access ON.

GPU: **런타임 → 런타임 유형 변경 → A100** (또는 L4).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 설정 변수

In [ ]:
REPO_URL = 'https://github.com/geniemo/osai.git'
BRANCH   = 'improve'
CONFIG   = 'p2/configs/d256.yaml'
DRIVE    = '/content/drive/MyDrive/osai/p2'         # 데이터 zip + ckpt 보관 위치
RUN_DIR  = f'{DRIVE}/runs/d256_main'                 # ckpt 저장 (Drive 직접)
DATA_DRIVE = f'{DRIVE}/train_50k_256.zip'
VALID_DRIVE = f'{DRIVE}/valid_10k_256.zip'
DATA_LOCAL = '/content/p2_data/train_50k_256.zip'    # 학습 throughput 위해 Colab 로컬

import os
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs('/content/p2_data', exist_ok=True)

## 2. 저장소 fresh clone

매 세션 처음부터 clone — Drive sync 없음, GitHub HEAD와 일치 보장.

In [ ]:
%cd /content
!rm -rf osai
!git clone --branch {BRANCH} --depth 1 {REPO_URL} osai
%cd osai
!git rev-parse --short HEAD

## 3. 의존성 설치

In [ ]:
!pip install -q pyyaml wandb pytorch-fid onnx onnxruntime scipy

## 4. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 5. WandB 로그인 (Colab Secret)

In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
print('WANDB_API_KEY loaded')

## 6. 학습 데이터 Colab 로컬로 복사

Drive over FUSE는 random-access I/O가 매우 느려 학습 throughput에 치명적 → zip을 Colab 로컬 SSD로 1회 cp.

In [ ]:
import os, shutil
if not os.path.exists(DATA_LOCAL):
    print(f'Copying {DATA_DRIVE} → {DATA_LOCAL} (~1-2 min)...')
    shutil.copy(DATA_DRIVE, DATA_LOCAL)
print('Local zip:', os.path.getsize(DATA_LOCAL) / 1e9, 'GB')

## 7. 학습 launch (first session)

- ckpt는 `{RUN_DIR}` (Drive)에 200k images마다 저장 → Disconnect 시 자동 보존
- WandB project `ffhqgen-skku-p2`, run `d256-stylegan2-scratch`
- ETA A100 ~22h. 24h session 끝나면 Resume 셀로 이어 학습.

In [ ]:
%cd /content/osai
!PYTHONPATH=. python p2/train.py \
    --config {CONFIG} \
    --train-zip {DATA_LOCAL} \
    --run-dir {RUN_DIR} 2>&1 | tee -a {RUN_DIR}/train.log

## 8. Resume — after disconnect

**Disconnect 후**: 셀 2 (Drive mount), 4 (clone), 5 (deps), 7 (wandb), 8 (데이터 cp) 다시 실행 후 이 셀.

Drive에 보존된 최신 ckpt를 자동 탐색해 `--resume`.

In [ ]:
import glob
ckpts = sorted(glob.glob(f'{RUN_DIR}/ckpt_*.pt'))
latest = ckpts[-1] if ckpts else None
print('Latest ckpt:', latest)
if latest:
    %cd /content/osai
    !PYTHONPATH=. python p2/train.py \
        --config {CONFIG} \
        --train-zip {DATA_LOCAL} \
        --run-dir {RUN_DIR} \
        --resume {latest} 2>&1 | tee -a {RUN_DIR}/train.log

## 9. FID 자기측정 (between sessions)

valid 통계 1회 캐시, 이후 매 측정. pytorch-fid CLI는 두 positional path 다음에 `--save-stats` 플래그가 와야 함.

In [ ]:
# One-time: valid zip → local dir, real-stats 캐시
import os, shutil
VALID_LOCAL = '/content/p2_data/valid_10k_256.zip'
VALID_DIR = '/content/p2_data/valid_10k_256_dir'
STATS = f'{DRIVE}/fid_stats_256.npz'

if not os.path.exists(VALID_LOCAL):
    print(f'Copying {VALID_DRIVE} → {VALID_LOCAL}...')
    shutil.copy(VALID_DRIVE, VALID_LOCAL)
if not os.path.isdir(VALID_DIR) or len(os.listdir(VALID_DIR)) == 0:
    os.makedirs(VALID_DIR, exist_ok=True)
    !cd {VALID_DIR} && unzip -q -o {VALID_LOCAL}
print('Valid dir:', VALID_DIR, '— files:', len(os.listdir(VALID_DIR)))

if not os.path.exists(STATS):
    # pytorch-fid: positional paths come FIRST, then --save-stats flag
    !python -m pytorch_fid {VALID_DIR} {STATS} --save-stats
assert os.path.exists(STATS), f'FID stats build failed — {STATS} not created'
print('FID stats ready:', STATS, '(', os.path.getsize(STATS) / 1024, 'KB )')

In [ ]:
# 최신 ckpt FID 측정 (final.pt 우선, 없으면 가장 큰 ckpt_*.pt)
import glob, os
candidates = []
if os.path.exists(f'{RUN_DIR}/final.pt'):
    candidates.append(f'{RUN_DIR}/final.pt')
candidates += sorted(glob.glob(f'{RUN_DIR}/ckpt_*.pt'))
latest = candidates[-1] if candidates else None
print('Evaluating:', latest)
if latest:
    %cd /content/osai
    !PYTHONPATH=. python p2/eval_fid.py \
        --ckpt {latest} \
        --stats {STATS} \
        --n 8000 --batch 32

## 10. ONNX export (리더보드 제출용)

`final.pt`가 있으면 그것 우선, 없으면 가장 큰 `ckpt_*.pt` 자동 선택.

In [ ]:
import glob, os
candidates = []
if os.path.exists(f'{RUN_DIR}/final.pt'):
    candidates.append(f'{RUN_DIR}/final.pt')
candidates += sorted(glob.glob(f'{RUN_DIR}/ckpt_*.pt'))
latest = candidates[-1] if candidates else None
print('Exporting:', latest)
assert latest, f'no ckpt found in {RUN_DIR}'
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
%cd /content/osai
!PYTHONPATH=. python p2/export_onnx.py \
    --ckpt {latest} \
    --out {DRIVE}/checkpoints/model.onnx
# model.onnx is now in MyDrive/osai/p2/checkpoints/ — Drive UI에서 다운로드